# Session 27: Decision Tree Regression Baseline 
**Week 4 — Decision Tree Regression Baseline** 
 
This notebook trains and evaluates a Decision Tree regressor. 
 
The model uses the same full-information training and test split. 
 
Test performance is measured using: 
- Mean Absolute Error (MAE) 
- Root Mean Squared Error (RMSE) 
- R-squared 
 
The primary model-ranking metric is RMSE. Lower RMSE indicates 
smaller prediction errors.

## 1. Imports and Project Paths

In [ ]:
from pathlib import Path 
import pickle 
import warnings 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from IPython.display import display 
from sklearn.tree import DecisionTreeRegressor 
from sklearn.metrics import ( 
    mean_absolute_error, 
    mean_squared_error, 
    r2_score, 
) 
from sklearn.model_selection import train_test_split 
 
RANDOM_STATE = 42 
TEST_SIZE = 0.20 
 
def find_project_root(start: Path | None = None) -> Path: 
    start = (start or Path.cwd()).resolve() 
    for candidate in [start, *start.parents]: 
        if (candidate / ".git").exists(): 
            return candidate 
    return start 
 
PROJECT_ROOT = find_project_root() 
DATA_DIRECTORY = PROJECT_ROOT / "data" 
PROCESSED_DIRECTORY = DATA_DIRECTORY / "processed" 
REPORTS_DIRECTORY = PROJECT_ROOT / "reports" 
TABLES_DIRECTORY = REPORTS_DIRECTORY / "tables" 
TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True) 
 
print("Project root:", PROJECT_ROOT) 
print("Processed-data directory:", PROCESSED_DIRECTORY) 
print("Output-table directory:", TABLES_DIRECTORY)

## 2. Load the Shared Full-Information Train/Test Split

In [ ]:
REQUIRED_SPLIT_KEYS = { 
    "Xtr_f", 
    "Xte_f", 
    "ytr", 
    "yte", 
} 
 
def _as_feature_frame(values, prefix: str = "feature") -> pd.DataFrame: 
    if isinstance(values, pd.DataFrame): 
        return values.reset_index(drop=True).copy() 
    array = np.asarray(values) 
    if array.ndim != 2: 
        raise ValueError( 
            f"Expected a two-dimensional feature matrix; received shape {array.shape}." 
        ) 
    columns = [ 
        f"{prefix}_{index:03d}" 
        for index in range(array.shape[1]) 
    ] 
    return pd.DataFrame(array, columns=columns) 
 
def _as_target_series(values) -> pd.Series: 
    if isinstance(values, pd.Series): 
        return values.reset_index(drop=True).copy() 
    if isinstance(values, pd.DataFrame): 
        if values.shape[1] != 1: 
            raise ValueError( 
                "The target DataFrame must have exactly one column." 
            ) 
        return values.iloc[:, 0].reset_index(drop=True) 
    array = np.asarray(values).reshape(-1) 
    return pd.Series(array, name="G3") 
 
def _normalize_split(values: dict): 
    Xtr_f = _as_feature_frame(values["Xtr_f"]) 
    Xte_f = _as_feature_frame(values["Xte_f"]) 
    ytr = _as_target_series(values["ytr"]) 
    yte = _as_target_series(values["yte"]) 
 
    if Xtr_f.shape[1] != Xte_f.shape[1]: 
        raise ValueError( 
            "Training and test feature matrices have different numbers of columns." 
        ) 
    if Xtr_f.shape[0] != len(ytr): 
        raise ValueError( 
            "Training feature and target row counts differ." 
        ) 
    if Xte_f.shape[0] != len(yte): 
        raise ValueError( 
            "Test feature and target row counts differ." 
        ) 
 
    Xte_f.columns = Xtr_f.columns 
    return Xtr_f, Xte_f, ytr, yte 
 
def load_split_from_npz(): 
    if not DATA_DIRECTORY.exists(): 
        return None 
    for path in sorted(DATA_DIRECTORY.rglob("*.npz")): 
        try: 
            with np.load(path, allow_pickle=True) as archive: 
                if REQUIRED_SPLIT_KEYS.issubset(archive.files): 
                    values = { 
                        key: archive[key] 
                        for key in REQUIRED_SPLIT_KEYS 
                    } 
                    print("Loaded train/test split from:", path) 
                    return _normalize_split(values) 
        except Exception: 
            continue 
    return None 
 
def load_split_from_pickle(): 
    if not DATA_DIRECTORY.exists(): 
        return None 
    pickle_paths = [ 
        *DATA_DIRECTORY.rglob("*.pkl"), 
        *DATA_DIRECTORY.rglob("*.pickle"), 
    ] 
    for path in sorted(pickle_paths): 
        try: 
            with path.open("rb") as file: 
                values = pickle.load(file) 
                if ( 
                    isinstance(values, dict) 
                    and REQUIRED_SPLIT_KEYS.issubset(values) 
                ): 
                    print("Loaded train/test split from:", path) 
                    return _normalize_split(values) 
        except Exception: 
            continue 
    return None 
 
def _read_array_file(path: Path): 
    suffix = path.suffix.lower() 
    if suffix == ".npy": 
        return np.load(path, allow_pickle=True) 
    if suffix == ".csv": 
        return pd.read_csv(path) 
    if suffix == ".parquet": 
        return pd.read_parquet(path) 
    raise ValueError(f"Unsupported array file: {path}") 
 
def load_split_from_separate_files(): 
    if not DATA_DIRECTORY.exists(): 
        return None 
    supported_suffixes = { 
        ".npy", 
        ".csv", 
        ".parquet", 
    } 
    file_index = {} 
    for path in DATA_DIRECTORY.rglob("*"): 
        if path.is_file() and path.suffix.lower() in supported_suffixes: 
            file_index.setdefault(path.stem.lower(), []).append(path) 
 
    aliases = { 
        "Xtr_f": ["xtr_f", "x_train_full", "xtrain_full"], 
        "Xte_f": ["xte_f", "x_test_full", "xtest_full"], 
        "ytr": ["ytr", "y_train", "ytrain"], 
        "yte": ["yte", "y_test", "ytest"], 
    } 
 
    resolved = {} 
    for required_name, possible_stems in aliases.items(): 
        match = None 
        for stem in possible_stems: 
            paths = file_index.get(stem.lower(), []) 
            if paths: 
                match = sorted(paths)[0] 
                break 
        if match is None: 
            return None 
        resolved[required_name] = match 
 
    values = { 
        key: _read_array_file(path) 
        for key, path in resolved.items() 
    } 
    print("Loaded separate train/test files:") 
    for key, path in resolved.items(): 
        print(f"  {key}: {path}") 
    return _normalize_split(values) 
 
def _read_table(path: Path) -> pd.DataFrame: 
    if path.suffix.lower() == ".csv": 
        return pd.read_csv(path) 
    if path.suffix.lower() == ".parquet": 
        return pd.read_parquet(path) 
    raise ValueError(f"Unsupported table format: {path}") 
 
def load_split_from_processed_table(): 
    if not PROCESSED_DIRECTORY.exists(): 
        return None 
    candidates = [ 
        *PROCESSED_DIRECTORY.rglob("*.parquet"), 
        *PROCESSED_DIRECTORY.rglob("*.csv"), 
    ] 
    excluded_terms = { 
        "comparison", 
        "prediction", 
        "result", 
        "metric", 
        "summary", 
        "correlation", 
    } 
    usable_candidates = [] 
    for path in candidates: 
        lowered_name = path.name.lower() 
        if any(term in lowered_name for term in excluded_terms): 
            continue 
        usable_candidates.append(path) 
 
    preferred_terms = [ 
        "full", 
        "encoded", 
        "processed", 
        "model", 
        "student", 
    ] 
 
    def candidate_score(path: Path): 
        name = path.name.lower() 
        score = sum( 
            1 
            for term in preferred_terms 
            if term in name 
        ) 
        return (-score, str(path).lower()) 
 
    for path in sorted(usable_candidates, key=candidate_score): 
        try: 
            table = _read_table(path) 
            target_column = next( 
                ( 
                    column 
                    for column in table.columns 
                    if str(column).strip().lower() == "g3" 
                ), 
                None, 
            ) 
            if target_column is None: 
                continue 
 
            y = pd.to_numeric( 
                table[target_column], 
                errors="raise", 
            ) 
            X = table.drop(columns=[target_column]).copy() 
 
            X = pd.get_dummies( 
                X, 
                drop_first=True, 
                dtype=float, 
            ) 
            X = X.apply( 
                pd.to_numeric, 
                errors="raise", 
            ) 
 
            if X.empty: 
                continue 
 
            Xtr_f, Xte_f, ytr, yte = train_test_split( 
                X, 
                y, 
                test_size=TEST_SIZE, 
                random_state=RANDOM_STATE, 
            ) 
            print("Created shared split from processed table:", path) 
            return _normalize_split( 
                { 
                    "Xtr_f": Xtr_f, 
                    "Xte_f": Xte_f, 
                    "ytr": ytr, 
                    "yte": yte, 
                } 
            ) 
        except Exception: 
            continue 
    return None 
 
split = ( 
    load_split_from_npz() 
    or load_split_from_pickle() 
    or load_split_from_separate_files() 
    or load_split_from_processed_table() 
) 
 
if split is None: 
    raise FileNotFoundError( 
        "No usable full-information train/test split or processed " 
        "table containing G3 was found under data/. Run the earlier " 
        "data-preparation sessions before executing this notebook." 
    ) 
 
Xtr_f, Xte_f, ytr, yte = split 
 
print() 
print("Training features:", Xtr_f.shape) 
print("Test features: ", Xte_f.shape) 
print("Training targets: ", ytr.shape) 
print("Test targets: ", yte.shape)

## 3. Validate the Modeling Arrays

In [ ]:
assert Xtr_f.shape[0] == len(ytr) 
assert Xte_f.shape[0] == len(yte) 
assert Xtr_f.shape[1] == Xte_f.shape[1] 
assert list(Xtr_f.columns) == list(Xte_f.columns) 
 
target_like_columns = { 
    str(column).strip().lower() 
    for column in Xtr_f.columns 
    if str(column).strip().lower() == "g3" 
} 
assert not target_like_columns, ( 
    "Target leakage detected: G3 appears among the features." 
) 
 
training_array = Xtr_f.to_numpy(dtype=float) 
test_array = Xte_f.to_numpy(dtype=float) 
 
assert np.isfinite(training_array).all() 
assert np.isfinite(test_array).all() 
assert np.isfinite(np.asarray(ytr, dtype=float)).all() 
assert np.isfinite(np.asarray(yte, dtype=float)).all() 
 
print("Modeling-array validation passed.")

## 4. Regression Evaluation Helper

In [ ]:
def eval_reg(y_true, y_pred) -> dict[str, float]: 
    mse = mean_squared_error(y_true, y_pred) 
    return { 
        "MAE": float( 
            mean_absolute_error(y_true, y_pred) 
        ), 
        "RMSE": float( 
            np.sqrt(mse) 
        ), 
        "R2": float( 
            r2_score(y_true, y_pred) 
        ), 
    } 
 
print("Regression evaluation helper is ready.")

## 5. Define and Train Decision Tree Regressor

In [ ]:
tree = DecisionTreeRegressor( 
    max_depth=5, 
    random_state=42, 
) 
 
print("Decision Tree Regressor:") 
print(tree)

## 6. Fit and Evaluate Decision Tree

In [ ]:
with warnings.catch_warnings(): 
    warnings.simplefilter("always") 
    tree.fit(Xtr_f, ytr) 
 
tree_predictions = tree.predict(Xte_f) 
tree_metrics = eval_reg(yte, tree_predictions) 
 
print("=" * 60) 
print("Decision Tree Regression Results") 
print(f"MAE: {tree_metrics['MAE']:.4f}") 
print(f"RMSE: {tree_metrics['RMSE']:.4f}") 
print(f"R2: {tree_metrics['R2']:.4f}") 
print("=" * 60)

## 7. Feature Importance (Decision Tree)

In [ ]:
# Get feature importances from the decision tree 
importances = tree.feature_importances_ 
feature_names = Xtr_f.columns 
 
# Create a DataFrame of feature importances 
importance_df = pd.DataFrame({ 
    'feature': feature_names, 
    'importance': importances 
}).sort_values('importance', ascending=False) 
 
# Show top 10 features 
print("Top 10 Most Important Features (Decision Tree):") 
display(importance_df.head(10)) 
 
# Plot feature importances 
plt.figure(figsize=(10, 6)) 
plt.barh(importance_df.head(10)['feature'], importance_df.head(10)['importance']) 
plt.xlabel('Importance') 
plt.title('Top 10 Feature Importances (Decision Tree)') 
plt.gca().invert_yaxis() 
plt.tight_layout() 
plt.show()

## 8. Compare with Session 25 Baselines

In [ ]:
# Load the Session 25 baseline results if they exist 
comparison_path = TABLES_DIRECTORY / "model_comparison_table.csv" 
 
if comparison_path.exists(): 
    comparison_table = pd.read_csv(comparison_path) 
    print("Existing model comparison table loaded.") 
else: 
    comparison_table = pd.DataFrame() 
    print("No existing comparison table found. Creating new one.") 
 
# Add Decision Tree results 
tree_row = { 
    "Session": 27, 
    "Week": 4, 
    "Task": "Regression", 
    "Scenario": "Full-information", 
    "Feature Set": "X_full", 
    "Target": "G3", 
    "Model": "Decision Tree", 
    "MAE": tree_metrics["MAE"], 
    "RMSE": tree_metrics["RMSE"], 
    "R2": tree_metrics["R2"], 
} 
 
# Remove any existing Decision Tree rows 
if not comparison_table.empty: 
    comparison_table = comparison_table[ 
        comparison_table["Model"] != "Decision Tree" 
    ].copy() 
 
# Append the new row 
comparison_table = pd.concat( 
    [comparison_table, pd.DataFrame([tree_row])], 
    ignore_index=True 
) 
 
# Save the updated comparison table 
comparison_table.to_csv(comparison_path, index=False) 
print(f"Decision Tree results saved to: {comparison_path}") 
 
# Show the comparison 
display(comparison_table.sort_values('RMSE').round(4))

## 9. Reflection 
Decision Tree Regression is a non-parametric model that can capture nonlinear relationships. 
 
Key advantages of Decision Trees: 
- Handles nonlinear relationships well 
- Provides feature importance rankings 
- Highly interpretable (can visualize the tree structure) 
- No need for feature scaling 
 
Potential limitations: 
- Prone to overfitting without proper depth control 
- Less stable than ensemble methods 
- May not generalize well if the tree is too deep 
 
Note: The `max_depth=5` parameter limits tree depth to prevent overfitting. 
 
Consider comparing Decision Tree performance to the baseline models and Random Forest to understand the trade-offs between interpretability and performance.

## 10. Completion Check

In [ ]:
# Verify all required components are present 
assert 'tree' in locals() 
assert 'tree_predictions' in locals() 
assert 'tree_metrics' in locals() 
assert all(k in tree_metrics for k in ['MAE', 'RMSE', 'R2']) 
assert 'importances' in locals() 
assert tree.max_depth == 5 
 
print("Session 27 notebook validation passed.") 
print(f"Decision Tree RMSE: {tree_metrics['RMSE']:.4f}") 
print(f"Decision Tree R2: {tree_metrics['R2']:.4f}")